# Aksara OCR — transformer seeds (complete the ±)

swin_tiny and vit_tiny were run at seed 0 only. This adds seeds 1 and 2 so the
transformer rows get a proper mean ± std.

**Cost — swin needs two sessions.** vit ~1.6 h/run (seeds 1,2 ≈ 3.2 h);
swin ~5.1 h/run (seeds 1,2 ≈ 10 h). With `--time-budget 8`, one commit finishes
vit and one swin seed; a second commit finishes the last swin seed.

## Setup & resume
1. **T4 x2**, **Internet On**, **Save & Run All (Commit)**.
2. **+ Add Input → Your Work →** the previous transformer/backbone output, so
   seed 0 (and anything already done) is restored and skipped.

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU. Settings (right sidebar) > Accelerator > GPU T4 x2, then rerun."
)

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
arch = f"sm_{major}{minor}"
supported = [a for a in torch.cuda.get_arch_list() if a.startswith("sm_")]

print(f"{name}  ({arch})")
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM")
print(f"torch {torch.__version__}  supports: {supported}")

# torch.cuda.is_available() returns True even when this build ships no kernels
# for the device - the failure then surfaces as a warning storm with every run
# landing in failures.jsonl. Check the architecture explicitly and stop here.
if arch not in supported:
    raise SystemExit(
        f"{name} is {arch}, but this PyTorch build only has kernels for "
        f"{supported}. Switch Settings > Accelerator to GPU T4 x2 (sm_75) "
        f"and rerun. The P100 is sm_60 and will not work."
    )

# Prove a real kernel runs, not just that a device is listed.
probe = (torch.randn(512, 512, device="cuda") @ torch.randn(512, 512, device="cuda")).sum()
torch.cuda.synchronize()
print(f"GPU compute OK (probe={probe.item():.1f})")

In [ ]:
# Internet must be ON (Settings > Internet) for these three lines.
import os
from pathlib import Path

REPO_URL = "https://github.com/phoenixfin/aksantara-ocr.git"
REPO = Path("/kaggle/working/aksantara-ocr")

if REPO.exists():
    !cd {REPO} && git pull -q
else:
    !git clone -q {REPO_URL} {REPO}

os.chdir(REPO)
# torch/torchvision ship with Kaggle; installing the rest avoids a slow reinstall
# of torch against a possibly-mismatched CUDA build.
!pip install -q timm pyyaml scikit-image tabulate
print(f"ready: {Path.cwd()}")

In [ ]:
# Restore finished transformer runs so only the missing seeds train.
import shutil
from pathlib import Path
ARTIFACTS = Path("/kaggle/working/artifacts")
RESULTS = ARTIFACTS / "results" / "backbones"
RESULTS.mkdir(parents=True, exist_ok=True)
sources  = list(REPO.glob("artifacts_kaggle/**/results/backbones"))
sources += list(Path("/kaggle/input").glob("*/artifacts/results/*"))
restored = 0
for cand in sources:
    if not cand.is_dir(): continue
    for run in cand.iterdir():
        if not run.is_dir() or not (run/"result.json").exists(): continue
        if "__unified__" not in run.name: continue
        dest = RESULTS/run.name
        if not dest.exists():
            shutil.copytree(run, dest); restored += 1
done = sorted(p.parent.name for p in RESULTS.glob("*vit_tiny*/result.json"))
done += sorted(p.parent.name for p in RESULTS.glob("*swin_tiny*/result.json"))
print(f"restored {restored} run(s); transformer runs present:")
for d in done: print("  ", d)

In [ ]:
# Fetch the cleaned, published v3. ~808 MB; needs Internet ON.
DOI = "10.17632/vfj32bpjsf.3"
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --list-only
!python scripts/00_fetch_mendeley.py --doi "{DOI}" --out /kaggle/working/raw

RAW_ROOT = "/kaggle/working/raw" 

In [ ]:
# Pre-resize once to 224px. Source images run up to 1500x1500; decoding one
# costs ~6.7 ms/core, so without this the dataloader, not the GPU, is the limit.
# A 224px cache drops that to ~1 ms/img.
!python scripts/00b_build_cache.py \
    --data-root "{RAW_ROOT}" --out /kaggle/working/data --size 224

DATA_ROOT = "/kaggle/working/data"
# Free the disk — the full-size tree is not needed again this session.
!rm -rf {RAW_ROOT}

In [ ]:
# stratified (no writer ids); --drop-duplicates removes images that became
# byte-identical after the 224px resize.
!python scripts/01_prepare_data.py \
    --data-root "{DATA_ROOT}" --out-dir "{ARTIFACTS}" \
    --split-strategy stratified --drop-duplicates

# Verify against published v3. Image/class/script counts come from manifest.csv,
# written before any filtering, so they are invariant.
import pandas as pd
manifest = pd.read_csv(f"{ARTIFACTS}/manifest.csv")
EXPECTED = {"images": 97383, "classes": 889, "scripts": 13}
actual = {"images": len(manifest), "classes": manifest["label"].nunique(),
          "scripts": manifest["script"].nunique()}
for k, want in EXPECTED.items():
    print(f"  {k:8} {actual[k]:6}  expected {want:6}  {'OK' if actual[k]==want else 'MISMATCH'}")
if actual != EXPECTED:
    raise SystemExit("Data does not match published v3 — re-run fetch and cache cells.")
print("Matches published v3.")

In [ ]:
# Runs backbone_transformer.yaml (seeds 0,1,2). Seed 0 is restored above and
# skips; seeds 1,2 train. --time-budget 8 stops cleanly before the batch limit,
# so if swin does not finish, attach this output to a new commit and rerun.
!python scripts/02_run_matrix.py --config configs/backbone_transformer.yaml     --artifacts "{ARTIFACTS}" --results "{RESULTS}" --num-workers 4 --time-budget 8

In [ ]:
import json
from pathlib import Path
from collections import defaultdict
from statistics import mean, stdev
g=defaultdict(list)
for f in Path(RESULTS).glob("*/result.json"):
    d=json.load(open(f)); e=d["experiment"]
    if e["model"] in ("vit_tiny","swin_tiny"):
        g[e["model"]].append(float(d["test_metrics"]["macro_f1"])*100)
for m,v in g.items():
    sd=f"±{stdev(v):.2f}" if len(v)>1 else "(1 seed)"
    print(f"{m:12} {mean(v):.2f} {sd}  (n={len(v)})")

When both have 3 seeds, paste me the numbers and I will update the backbone table and figure.